### KIKO_Real-Time Detection and Filtering of Abusive Content

Feature extraction


In [10]:
import pandas as pd
import numpy as np

In [11]:
# Load and harmonize all cleaned training datasets
from pathlib import Path

def to_common_schema(frame: pd.DataFrame) -> pd.DataFrame:
    f = frame.copy()
    cols_lower = {c.lower(): c for c in f.columns}

    if 'text' not in f.columns:
        if 'review' in cols_lower:
            f = f.rename(columns={cols_lower['review']: 'text'})
        elif 'tweet' in cols_lower:
            f = f.rename(columns={cols_lower['tweet']: 'text'})
        elif 'Text' in f.columns:
            f = f.rename(columns={'Text': 'text'})

    if 'sentiment' not in f.columns:
        if 'Sentiment' in f.columns:
            f = f.rename(columns={'Sentiment': 'sentiment'})

    if {'text', 'sentiment'}.issubset(f.columns):
        return f[['text', 'sentiment']]
    return pd.DataFrame(columns=['text', 'sentiment'])

files = [
    r'D:\ML CODES\IBM\Train_Cleaned\Airline_train_cleaned.csv',
    r'D:\ML CODES\IBM\Train_Cleaned\text.tweeet_cleaned.csv',
    r'D:\ML CODES\IBM\Train_Cleaned\train_tweet_cleaned.csv',
    r'D:\ML CODES\IBM\Train_Cleaned\train_cleaned.csv',
    r'D:\ML CODES\IBM\Train_Cleaned\IMDB_Dataset_cleaned.csv',
    r'D:\ML CODES\IBM\Train_Cleaned\Sentiment_train_cleaned.csv'
    ]

frames = []
for fp in files:
    if Path(fp).exists():
        frames.append(to_common_schema(pd.read_csv(fp)))

# Concatenating dataframes
df = pd.concat(frames, ignore_index=True)

In [91]:
df.head()

,sentiment,text
0,neutral,What said
1,positive,plus youve added commercials to the experienc...
2,neutral,I didnt today Must mean I need to take anothe...
3,negative,its really aggressive to blast obnoxious ente...
4,negative,and its a really big bad thing about it


In [92]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45463 entries, 0 to 45462
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  45463 non-null  object
 1   text       45462 non-null  object
dtypes: object(2)
memory usage: 710.5+ KB


In [93]:
print(df['sentiment'].value_counts())

sentiment
negative    17869
neutral     15612
positive    11982
Name: count, dtype: int64


In [94]:
df.duplicated().sum()

285

In [12]:
df.drop_duplicates(inplace=True)

In [13]:
df.dropna(inplace=True)

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [15]:
# Normalize labels, map variants, and keep only 3 classes
label_map = {
    'neg': 'negative', 'negative': 'negative',
    'neu': 'neutral', 'neutral': 'neutral',
    'pos': 'positive', 'positive': 'positive'
}

df['sentiment'] = (
    df['sentiment']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(label_map)
    )

df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].ne('') & df['sentiment'].notna()]

x = df['text']
y = df['sentiment']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

print('Rows used:', len(df))
print('Class counts:\n', y.value_counts())

Rows used: 154785
Class counts:
 sentiment
positive    74754
negative    62838
neutral     17193
Name: count, dtype: int64


In [ ]:
# Word + character TF-IDF features with logistic regression
pipeline = Pipeline([
    ('features', FeatureUnion([
        ('word', TfidfVectorizer(sublinear_tf=True, stop_words='english')),
        ('char', TfidfVectorizer(analyzer='char_wb'))
    ])),
    ('clf', LogisticRegression(max_iter=4000, n_jobs=-1))
])

param_grid = {
    'features__word__ngram_range': [(1, 2)],
    'features__word__min_df': [2, 3],
    'features__word__max_df': [0.9, 0.95],
    'features__char__ngram_range': [(3, 5), (3, 6)],
    'features__char__min_df': [2],
    'features__char__max_df': [0.95],
    'clf__C': [1.0, 2.0, 4.0],
    'clf__class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    verbose=2,
    scoring='accuracy'
    )

grid_search.fit(x_train, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(x_test)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 48 candidates, totalling 144 fits


In [9]:
# Quick audit of available cleaned training files
import glob
from pathlib import Path

for fp in sorted(glob.glob(r'D:\ML CODES\IBM\Train_Cleaned\*.csv')):
    name = Path(fp).name
    try:
        t = pd.read_csv(fp)
        cols = list(t.columns)
        if {'text', 'sentiment'}.issubset(t.columns):
            vc = t['sentiment'].astype(str).str.lower().value_counts().head(5).to_dict()
            print(f"{name:30} rows={len(t):7d}  labels={vc}")
        else:
            print(f"{name:30} rows={len(t):7d}  columns={cols}")
    except Exception as e:
        print(name, '->', e)

Airline_train_cleaned.csv      rows=  14452  labels={'negative': 9087, 'neutral': 3067, 'positive': 2298}
IMDB_Dataset_cleaned.csv       rows=  49582  columns=['review', 'sentiment']
Sentiment_train_cleaned.csv    rows=    708  columns=['Sentiment', 'Text']
text.tweeet_cleaned.csv        rows=   3531  labels={'neutral': 1428, 'positive': 1102, 'negative': 1001}
train_cleaned.csv              rows=  60000  labels={'positive': 38029, 'negative': 20292, 'neutral': 1679}
train_tweet_cleaned.csv        rows=  27480  labels={'neutral': 11117, 'positive': 8582, 'negative': 7781}
